In [ ]:
import geopandas as gpd
import pandas as pd

import cafo_iowa.db.session as s
import cafo_iowa.db.models as m

from cafo_iowa.data.helpers.geo import get_clusters
from cafo_iowa.utils.visualize import simple_map


import matplotlib.pyplot as plt
import seaborn as sns
import contextily as ctx

In [ ]:
session = s.get_session()
engine = session.get_bind()

In [ ]:
query = f"""
WITH labelled_tiles AS (
    SELECT DISTINCT jsonb_array_elements_text(naip_qt_ids) AS tile_id
    FROM processed.{m.LabelBatches.__tablename__}
),
permit_tiles AS (
    SELECT 
        naip_qt_id,
        COUNT(*) AS n_permits
    FROM processed.{m.Permits.__tablename__}
    WHERE animal_units > 0
    GROUP BY naip_qt_id
)

SELECT 
    nq.*, 
    (lt.tile_id IS NOT NULL) AS labelled,
    COALESCE(pt.n_permits, 0) AS n_permits
FROM 
    processed.{m.Naip21QT.__tablename__} nq
LEFT JOIN 
    labelled_tiles lt
ON 
    nq.id = lt.tile_id
LEFT JOIN
    permit_tiles pt
ON
    nq.id = pt.naip_qt_id;
"""

data = gpd.read_postgis(query, engine, geom_col="geometry")

In [ ]:
print("Number of tiles:", len(data))
print("Number of labelled tiles:", data["labelled"].sum())
print(
    "Number of labeled tiles with permits:",
    data[data.labelled & (data.n_permits > 0)].shape[0],
)

In [ ]:
num = data[(data.labelled == False) & (data.n_permits > 0)].shape[0]
print(f"Number of tiles with 1+ permits that haven't been labelled:", num)

In [ ]:
data["permit_cat"] = "Unlabeled, No permit"
data.loc[(data.n_permits > 0) & (data.labelled), "permit_cat"] = "Labeled, With permit"
data.loc[(data.n_permits > 0) & (~data.labelled), "permit_cat"] = (
    "Unlabeled, With permit"
)
data.loc[(data.n_permits == 0) & (data.labelled), "permit_cat"] = "Labeled, No permit"

In [ ]:
# Plot map of all tiles
fig, ax = plt.subplots(1, 1, figsize=(12, 12))

# Plot data with a custom color map and a legend
data.plot(
    ax=ax,
    column="permit_cat",
    cmap="viridis",
    legend=True,
)
# Add a basemap for geographic context
ctx.add_basemap(ax, crs=data.crs.to_string(), source=ctx.providers.CartoDB.Positron)

# Add a title
ax.set_title("Label status of tile", fontsize=16)

# Remove axis ticks and labels
ax.set_axis_off()

# Show the plot
plt.tight_layout()
plt.savefig("label_status.png", dpi=300)
plt.show()